# 1.message
## basemessage
- SystemMessage
- HumanMessage
- AIMessage
- ToolMessage

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
from langchain.tools import tool
import requests
# ==============================================
# 🌤️ 真实天气查询工具（高德地图 API）
# ==============================================
@tool
def get_weather(city: str, date: str = "明天") -> str:
    """
    查询中国城市的实时、未来天气
    参数:
    - city: 城市名，例如 北京、上海、杭州
    - date: 日期，例如 今天、明天、后天
    """
    try:
        # 高德地图天气 API（免费申请）
        AMAP_KEY = os.getenv("GAODE_TIANQI_API_KEY")  # 等下我教你免费拿
        url = "https://restapi.amap.com/v3/weather/weatherInfo"
        
        # 1. 先获取城市编码
        city_url = "https://restapi.amap.com/v3/config/district"
        city_params = {
            "keywords": city,
            "key": AMAP_KEY,
            "subdistrict": 0,
            "output": "json"
        }
        city_resp = requests.get(city_url, params=city_params).json()
        if not city_resp.get("districts"):
            return f"未找到城市：{city}"
        city_code = city_resp["districts"][0]["adcode"]

        # 2. 获取天气
        weather_params = {
            "city": city_code,
            "key": AMAP_KEY,
            "extensions": "all",  # 获取预报
            "output": "json"
        }
        resp = requests.get(url, params=weather_params).json()
        forecasts = resp["forecasts"][0]["casts"]

        # 3. 返回对应日期天气
        if date == "今天":
            data = forecasts[0]
        elif date == "明天":
            data = forecasts[1]
        elif date == "后天":
            data = forecasts[2]
        else:
            data = forecasts[1]

        return (
            f"{date} {city} 天气：{data['dayweather']}，"
            f"温度 {data['nighttemp']}~{data['daytemp']}℃，"
            f"风向：{data['daywind']}，风力：{data['daypower']}级"
        )
    except Exception as e:
        return f"天气查询失败：{str(e)}"


In [19]:
import os
from langchain_openai import ChatOpenAI
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
tools = [get_weather]
llm = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-plus",
    temperature=0,
)

prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content="你是我的人工智能助手，协助我查询天气信息。"),
        MessagesPlaceholder(variable_name="chat_history"),
        HumanMessage(content="你好，我是安"),
        AIMessage(content="你好，安！很高兴见到你。我是一个由DashScope提供支持的AI助手。有什么我可以帮助你的吗？"),
        HumanMessage(content="北京天气如何"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
        # ("system", "你是我的人工智能助手，协助我查询天气信息。"),
        # ("human", "你好，我是安"),
        # ("assistant", "你好，安！很高兴见到你。我是一个由DashScope提供支持的AI助手。有什么我可以帮助你的吗？"),
        # ("human", "北京天气如何"),
    ]
)
agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)
response = agent_executor.invoke({
    "input": "北京天气如何",
    "chat_history": []
})
print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'city': '北京', 'date': '今天'}`


今天 北京 天气：多云，温度 10~23℃，风向：南，风力：1-3级今天北京的天气是多云，气温在10℃到23℃之间，风向为南风，风力为1-3级。适合外出活动哦！如果需要了解其他日期或城市的天气，随时告诉我～

> Finished chain.
今天北京的天气是多云，气温在10℃到23℃之间，风向为南风，风力为1-3级。适合外出活动哦！如果需要了解其他日期或城市的天气，随时告诉我～


In [11]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from langchain.tools import tool
import os
@tool
def search_weather(city: str) -> str:
    """查询指定城市的天气情况。
    Args:
        city: 需要查询天气的城市名称，例如"北京"
    """
    return f"{city}今天天气晴朗，气温25℃"

tools = [search_weather] 
# 初始化模型
model = init_chat_model(
    model= "qwen3.6-plus",
    model_provider= "openai",
    base_url= "https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key= os.getenv("DASHSCOPE_API_KEY"),
    temperature=0,
)

agent = create_react_agent(model=model, tools=tools, prompt="You are a helpful assistant")
result = agent.invoke({"messages": [HumanMessage(content=[
    {
        "type": "image_url",
        "image_url": {
            "url": "https://hellorfimg.zcool.cn/provider_image/large/hi2246794331.jpg"
        }
    },
    {"type": "text", "text": "描述一下这张图片里有什么"},
])]})
print(result)

/Users/anshenggui/PycharmProjects/example-app/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


{'messages': [HumanMessage(content=[{'type': 'image_url', 'image_url': {'url': 'https://hellorfimg.zcool.cn/provider_image/large/hi2246794331.jpg'}}, {'type': 'text', 'text': '描述一下这张图片里有什么'}], additional_kwargs={}, response_metadata={}, id='bbcaced5-fd89-4346-9fb4-bbf340c94402'), AIMessage(content='这张图片展示了一座宏伟的**中国传统风格建筑**（看起来像是一座塔楼或阁楼），背景是晴朗的蓝天白云。以下是详细的画面内容描述：\n\n1.  **主体建筑**：\n    *   这是一座多层的木结构古建筑，从下往上可以看到明显的**四层屋檐**。\n    *   建筑风格具有典型的中式特色，拥有**飞檐翘角**，屋角像鸟翼一样高高翘起，线条优美。\n    *   屋顶覆盖着深灰色的瓦片。\n    *   建筑的柱子和门窗呈现深红褐色（或紫红色），显得古朴庄重。门窗上有精细的雕花格纹。\n    *   在建筑底层的正门上方，悬挂着一块黑底金字的**牌匾**，上面的汉字从右往左读似乎是“瞻四方”或者从左往右“四方瞻”（字迹稍显模糊）。\n\n2.  **自然环境**：\n    *   **天空**：占据了画面上半部分，呈现出鲜艳的湛蓝色，漂浮着许多白云，天气看起来非常好，光线充足。\n    *   **植被**：建筑周围绿树环绕。右侧有茂密的树木，前景右下角有一丛修剪整齐的绿色灌木。\n\n3.  **前景与细节**：\n    *   **台阶**：建筑前方有石砌的台阶通向大门。\n    *   **花坛**：图片左下角有两个长方形的花坛，分别种着黄色和蓝紫色的花朵，为画面增添了色彩。\n    *   **人物与车辆**：在图片左侧的树荫下，隐约可以看到一个人坐在长椅上休息，旁边还停放着白色的汽车。\n\n4.  **其他信息**：\n    *   图片上有“站酷海洛”的水印，说明这是一张版权图片。右下角有图片的ID编号。\n\n总的来说，这是一张构图精美、色

In [71]:
print(result["messages"][-1].content,end="",flush=True)

这张图片展示了一座宏伟的**中国传统风格楼阁**，矗立在蓝天白云之下。以下是图片中的主要元素：

1.  **主体建筑**：
    *   这是一座多层的木结构塔楼（看起来像是成都望江楼公园的望江楼，即崇丽阁）。
    *   建筑拥有典型的**飞檐翘角**设计，屋顶层层叠叠，覆盖着深灰色的瓦片，檐角高高翘起，显得轻盈而灵动。
    *   楼体主要由深红褐色（或紫红色）的木材构成，显得古朴庄重。
    *   每一层都有精美的木质窗棂和栏杆。
    *   底层入口处上方悬挂着一块黑底金字的牌匾（字迹较模糊，依稀可见汉字）。
    *   建筑底部有石砌的基座和台阶，台阶两侧有石栏杆，入口处似乎还有石狮子守护。

2.  **自然环境**：
    *   **天空**：背景是湛蓝的天空，飘浮着大朵洁白的云彩，天气非常晴朗，光线明亮。
    *   **植被**：建筑周围绿树环绕。右侧有茂密的绿色灌木和树木，左侧有一棵枝干较细的树。
    *   **花卉**：前景的花坛里种着鲜艳的黄色和蓝紫色花朵，为画面增添了生机。

3.  **其他细节**：
    *   **人物**：在图片左下角的长椅上，坐着一个人（看起来像是一位老人），正在休息或观赏风景，给画面增添了一丝生活气息。
    *   **车辆**：左侧远处的树荫下停放着几辆白色的汽车。
    *   **水印**：图片上有“站酷海洛”的水印，右下角有图片ID，表明这是一张版权图片。

总的来说，这是一张色彩鲜艳、构图精美的风景照，展现了中国古典建筑与自然美景的和谐融合。

In [73]:
stream = agent.stream({"messages": [HumanMessage([
    {
        "type": "image_url",
        "image_url": {
            "url": "https://hellorfimg.zcool.cn/provider_image/large/hi2246794331.jpg"
        }
    },
    {"type": "text", "text": "描述一下这张图片里有什么"},
])]},
stream_mode = "messages"
)
for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content,end="",flush=True)

这张图片展示了一座宏伟的**中国传统风格楼阁（或塔）**，矗立在蓝天白云之下。以下是图片中的主要元素：

1.  **主体建筑**：
    *   这是一座多层的木结构建筑，看起来有四层明显的屋檐。
    *   建筑风格古朴典雅，拥有典型的**飞檐翘角**（屋檐向上翘起），覆盖着深灰色的瓦片。
    *   建筑的主体颜色呈现深红褐色或紫红色，配有精美的木质窗棂和栏杆。
    *   在底层入口的正上方，悬挂着一块黑底金字的**牌匾**（虽然字迹稍显模糊，但能看出是传统的匾额）。
    *   建筑底部有石砌的基座和通往大门的石阶。

2.  **自然环境**：
    *   **天空**：背景是非常明亮的蓝天，飘浮着许多白云，显示天气非常晴朗。
    *   **植被**：建筑周围环绕着绿色的树木。前景右侧有茂密的灌木，左侧有一些较高的树木。
    *   **花卉**：图片最下方有两个花坛，分别种着黄色和蓝紫色的花朵，为画面增添了色彩。

3.  **其他细节**：
    *   **人物**：在图片左下角的树荫下，有一张长椅，上面坐着一个人（看起来像是在休息的老人）。
    *   **水印**：图片上有“站酷海洛”的水印，表明这是一张版权图片。

总的来说，这是一张构图精美的风景照，展现了一座古色古香的中国建筑在明媚阳光下的风貌，很可能是在某个公园或历史遗迹（如成都的望江楼公园）拍摄的。

In [1]:
from ipywidgets import FileUpload
from IPython.display import display
upload = FileUpload(accept=".jpg,.jpeg,.png,.gif,.bmp,.tiff",multiple=False)
display(upload)





FileUpload(value=(), accept='.jpg,.jpeg,.png,.gif,.bmp,.tiff', description='Upload')

In [7]:
print(upload.value)

({'name': 'u=2172818577,3783888802&fm=253&app=138&f=JPEG.jpeg', 'type': 'image/jpeg', 'size': 86739, 'content': <memory at 0x10bcddd00>, 'last_modified': datetime.datetime(2026, 4, 23, 12, 28, 29, 900000, tzinfo=datetime.timezone.utc)},)


In [8]:
import base64
uploaded_file = upload.value[0]
content_mv = uploaded_file['content']
img_bytes = bytes(content_mv)
img_base64 = base64.b64encode(img_bytes).decode('utf-8')

In [13]:
from langchain_core.messages import HumanMessage
multimodal_question = HumanMessage(content=[
    {
        "type": "image_url",
        "image_url": {
            "url": f"data:image/jpeg;base64,{img_base64}"
        }
    },
    { "type": "text", "text": "描述一下这张图片里有什么" }
])
stream = agent.stream(
    {"messages": [multimodal_question]},
    stream_mode = "messages"
)
for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content,end="",flush=True)

这张图片展示了一幅从飞机舷窗向外拍摄的壮丽高空景色。主要包含以下几个元素：

1.  **飞机部件（前景）**：图片的下方占据了显著位置的是飞机的机翼和发动机外壳。可以看到银灰色的金属表面，上面排列着整齐的铆钉。阳光照射在发动机光滑的曲面上，反射出耀眼的星芒状光辉。

2.  **云层（中景）**：在机翼上方，漂浮着大片洁白、蓬松的积云。云层看起来很厚，层层叠叠，像棉花糖一样铺散在空中。

3.  **雪山（远景）**：透过云层的缝隙，可以看到远处连绵起伏的雪山。山峰被皑皑白雪覆盖，呈现出冷峻的蓝白色调，看起来非常雄伟壮观，很有可能是喜马拉雅山脉或阿尔卑斯山脉等高海拔地区。

4.  **蓝天（背景）**：图片的最上方是纯净、深邃的蔚蓝色天空，与白色的云朵和雪山形成了鲜明的对比。

总的来说，这是一张典型的航拍风景照，展现了高空飞行时看到的云海与雪山交织的美景。